# 14 · Filter RAG passages before generation

## The problem

Retrieval-augmented generation (RAG) fetches passages and pastes them into the LLM prompt. But retrieval is matching
**words and vectors**, not meaning, so it happily returns passages that are *keyword-adjacent* and useless. Ask *"How do
I reset my router to factory settings?"* and you also get the router **factory** in Shenzhen, and how to reset your
**account password**. The LLM then answers from noise, hedges, or burns tokens reading junk.

## Why Jev

Put a **keep/drop gate** between retrieval and generation: for every passage, *does this actually help answer the
query?* That's a yes/no judgment, many times per query, on the hot path of every request. Jev is a **System One**
model: typed, **calibrated** probabilities in about 100 ms for a fraction of a cent, and **many questions fit in one
call**, so one request scores every passage at once.

Unlike the reranker in notebook 08 (which *orders* passages), this is a **filter**, and a filter has to be **measured**:
how many useful passages did we drop, and how much junk did we let through?

## What we'll build

```
query ──► retriever ──► 8 passages ──► Jev: ONE call, one Noul per passage ──► keep if p ≥ KEEP_AT ──► LLM prompt
                                                    │
                          labelled passages ────────┴──► measure precision / recall at many thresholds ──► pick KEEP_AT
```

## What you'll learn
- Building **one question per item** with a dict comprehension, all answered in a single call.
- Measuring a filter: **missed** useful passages (recall) vs **junk kept** (precision).
- The core lesson: **measure first, then pick the threshold.** 0.5 is not a magic number.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #6.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the Jev client, `jev`, that every later cell uses,
- imports the Jev question types (here we only need `Noul`),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so you can *see* the numbers.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · A query, the retrieved passages, and our labels

Pretend the retriever returned these eight passages for one query. Next to each one we wrote a **label** by hand:
`True` if it actually helps answer the question. Those labels are what lets us *measure* the filter later. In a real
project you'd label a few dozen queries like this; it's the most valuable hour you'll spend on a RAG system.

Four are useful (the button method, the admin-page method, and two things you should know before resetting). Four are
traps: they share the words *factory* or *reset* but don't help.

In [2]:
QUERY = "How do I reset my router to factory settings?"

PASSAGES = [
    ("Press and hold the recessed RESET button on the back for 10 seconds until the lights blink.", True),
    ("After a factory reset, the Wi-Fi name and password return to those printed on the label.", True),
    ("Our routers ship with factory-installed antennas for better range.", False),
    ("To reset your account password, click 'Forgot password' on the login page.", False),
    ("You can also restore defaults from the admin page: Settings > System > Factory Reset.", True),
    ("The router's factory is located in Shenzhen and is ISO 9001 certified.", False),
    ("Rebooting (unplugging for 30 seconds) fixes most slow-connection issues.", False),
    ("Resetting erases port forwarding rules, so note them down first.", True),
]
texts = [t for t, _ in PASSAGES]
labels = [g for _, g in PASSAGES]
print(f"{len(PASSAGES)} passages, {sum(labels)} labelled useful")

8 passages, 4 labelled useful


## Step 2 · One request, one `Noul` per passage

**How a `Noul` works:** `.noul` is P(yes), from 0 to 1. Here *yes* means "this passage helps answer the query".

`usefulness()` builds **one question per passage** (`p0`, `p1`, ...) with a dict comprehension and sends them **all in
one `system_one` call**. The `state` is a dict `{"query": ..., "passages": [...]}`, and the instruction refers to
`` `query` `` and `` `passages[3]` `` with backticks, which point into that dict.

This cell only **defines** the function. The next one calls it.

In [3]:
def usefulness(query, passages, client):
    """P(passage helps answer query) for every passage, in a single Jev call."""
    if "jev" not in globals():
        raise RuntimeError("jev is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")
    questions = {
        f"p{i}": Noul(instructions=f"Does `passages[{i}]` contain information that helps answer `query`?")
        for i in range(len(passages))
    }
    r = client.system_one(state={"query": query, "passages": passages}, questions=questions)
    return [r.nouls[f"p{i}"].noul for i in range(len(passages))]


print("usefulness() defined:", len(texts), "questions per call for this query")

usefulness() defined: 8 questions per call for this query


Now score the eight passages. We keep the probabilities in `probs` so the next steps can measure many thresholds
**without** calling Jev again.

In [4]:
probs = usefulness(QUERY, texts, jev)
for (t, gold), p in zip(PASSAGES, probs):
    print(f"  {'useful' if gold else '      '}  {bar(p)} {p:.2f}  {t[:64]}")

  useful  ████████████████████ 0.99  Press and hold the recessed RESET button on the back for 10 seco
  useful  ██████·············· 0.32  After a factory reset, the Wi-Fi name and password return to tho
          █··················· 0.03  Our routers ship with factory-installed antennas for better rang
          █··················· 0.03  To reset your account password, click 'Forgot password' on the l
  useful  ████████████████████ 0.98  You can also restore defaults from the admin page: Settings > Sy
          ···················· 0.02  The router's factory is located in Shenzhen and is ISO 9001 cert
          █··················· 0.05  Rebooting (unplugging for 30 seconds) fixes most slow-connection
  useful  ██████·············· 0.32  Resetting erases port forwarding rules, so note them down first.


**What to notice**
- The two *how-to* passages (the button, the admin page) score **near 1**.
- All four traps score **near 0**: Jev is not fooled by *factory* (antennas, Shenzhen) or *reset* (account password),
  and it knows rebooting is not resetting.
- The two other useful passages (Wi-Fi name returns to the label; port forwarding rules get erased) land in the **low
  middle**, roughly 0.3 to 0.4. They don't tell you *how* to reset, they tell you what happens when you do. Jev is
  honestly uncertain whether that "helps answer" the question.
- That middle zone is exactly where the threshold choice matters. Next, we measure it.

## Step 3 · Measure the filter at several thresholds

`measure()` compares the keep/drop decisions with our labels:

| metric | meaning | why it matters in RAG |
|---|---|---|
| **missed** | useful passages we dropped | the LLM never sees them, so the answer is incomplete |
| **junk kept** | useless passages we let through | noise in the prompt, wasted tokens, risk of off-topic answers |
| **precision** | of what we kept, the share that was useful | |
| **recall** | of what was useful, the share we kept | |

We also print the **highest score of any junk passage** and the **lowest score of any useful passage**. If the first is
below the second, there's a gap, and any threshold inside it is perfect on this data.

In [5]:
def measure(probs, labels, threshold):
    kept = [p >= threshold for p in probs]
    tp = sum(k and g for k, g in zip(kept, labels))
    missed = sum((not k) and g for k, g in zip(kept, labels))
    junk = sum(k and not g for k, g in zip(kept, labels))
    precision = tp / max(1, tp + junk)
    recall = tp / max(1, tp + missed)
    return {"kept": sum(kept), "missed": missed, "junk": junk, "precision": precision, "recall": recall}


for th in (0.1, 0.2, 0.5, 0.8, 0.95):
    m = measure(probs, labels, th)
    print(f"threshold {th:.2f}: kept {m['kept']}  missed {m['missed']}  junk {m['junk']}  "
          f"precision {m['precision']:.2f}  recall {m['recall']:.2f}")

max_junk = max(p for p, g in zip(probs, labels) if not g)
min_useful = min(p for p, g in zip(probs, labels) if g)
print(f"\nhighest junk score {max_junk:.2f}   lowest useful score {min_useful:.2f}   "
      f"-> gap {'from %.2f to %.2f' % (max_junk, min_useful) if max_junk < min_useful else 'NONE (classes overlap)'}")

threshold 0.10: kept 4  missed 0  junk 0  precision 1.00  recall 1.00
threshold 0.20: kept 4  missed 0  junk 0  precision 1.00  recall 1.00
threshold 0.50: kept 2  missed 2  junk 0  precision 1.00  recall 0.50
threshold 0.80: kept 2  missed 2  junk 0  precision 1.00  recall 0.50
threshold 0.95: kept 2  missed 2  junk 0  precision 1.00  recall 0.50

highest junk score 0.05   lowest useful score 0.32   -> gap from 0.05 to 0.32


**What to notice**
- At **0.5**, the "obvious default", precision is perfect but recall is only 0.50: we silently drop **2 of the 4** useful
  passages (the Wi-Fi label and the port-forwarding warning). The answer would tell the user how to reset, but not
  that they'll lose their port forwarding rules.
- At **0.2** (and 0.1) the filter is perfect on this query: all 4 useful passages kept, 0 junk.
- There is a clear **gap** between the highest junk score (near 0) and the lowest useful score (around 0.3). Any
  threshold inside that gap separates the two groups perfectly.
- Raising the threshold never adds junk here; it only costs recall. On other queries the trade-off can go both ways,
  which is why you measure.

## Step 4 · Pick the threshold from the measurement, then ship the filter

The measurement says the useful passages start around 0.3 and the junk tops out near 0.05. We pick **`KEEP_AT = 0.2`**:
- it sits **inside the gap**, with room on both sides, so small run-to-run wobbles in the scores don't flip decisions;
- it leans towards the **junk side**, on purpose. In RAG a missed useful passage is usually worse than one extra noisy
  passage: the LLM can ignore noise, but it can't use what it never saw.

> **Workshop note:** the first draft hard-coded `KEEP_AT = 0.5` and missed half the useful passages, even though its own
> measurement one cell earlier showed 0.2 was perfect. That's the lesson of this notebook: **measure, then pick the
> threshold**. And one query with eight passages is a demo, not an evaluation. Before production, label a few dozen
> real queries and re-run Step 3.

`filter_passages()` is the function your RAG pipeline calls: score, then keep only passages at or above `KEEP_AT`. The
cell then shows the context that would actually be pasted into the LLM prompt.

In [6]:
KEEP_AT = 0.2   # chosen from Step 3: inside the gap between junk (~0.05) and the weakest useful passage (~0.3)


def filter_passages(query, passages, client, keep_at=KEEP_AT):
    probs = usefulness(query, passages, client)
    return [(t, p) for t, p in zip(passages, probs) if p >= keep_at]


m = measure(probs, labels, KEEP_AT)
print(f"at KEEP_AT={KEEP_AT}: precision {m['precision']:.2f}  recall {m['recall']:.2f}  (from the cached scores)\n")

context = "\n".join(f"- {t}" for t, _ in filter_passages(QUERY, texts, jev))
print("CONTEXT SENT TO THE LLM:\n" + context)

at KEEP_AT=0.2: precision 1.00  recall 1.00  (from the cached scores)

CONTEXT SENT TO THE LLM:
- Press and hold the recessed RESET button on the back for 10 seconds until the lights blink.
- After a factory reset, the Wi-Fi name and password return to those printed on the label.
- You can also restore defaults from the admin page: Settings > System > Factory Reset.
- Resetting erases port forwarding rules, so note them down first.


**What to notice**
- The context now has all four useful passages: both reset methods **and** the two warnings.
- None of the four traps made it in, so the prompt is half the size it would have been with no filter.
- `filter_passages()` makes a fresh Jev call, so the scores can wobble slightly from Step 2. Because `KEEP_AT` sits in
  the middle of the gap, that wobble doesn't change what gets kept.

## Step 5 · Give it a face

Everything from here on is **presentation only**. This cell is a small style kit (colours, `box`, `badge`, `meter`) and
`form_app()`, which wires text boxes and a button to any function that returns HTML. Below it, a threshold slider re-runs `measure()` on the cached scores, and the try-it form filters your own passages.
**The decision logic above is untouched**: the UI only calls the same functions you've already seen run.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

Slide the threshold and watch passages switch between KEEP and drop, with the metrics updating live (cached scores, no
new calls). Start at 0.5, where two useful passages are flagged **missed useful**, then slide down to 0.2. Push it all
the way down to about 0.05 and junk starts getting kept.

In [8]:
def filter_board(threshold=KEEP_AT):
    m = measure(probs, labels, threshold)
    rows = ""
    for (t, gold), p in zip(PASSAGES, probs):
        keep = p >= threshold
        tag = badge("KEEP", SAGE) if keep else badge("drop", "#B9B4A8")
        warn = ""
        if keep and not gold:
            warn = f' <span style="color:{RED};font-size:12px;">junk kept</span>'
        if gold and not keep:
            warn = f' <span style="color:{RED};font-size:12px;">missed useful</span>'
        rows += f'<div style="padding:4px 0;opacity:{1 if keep else .5};">{tag} <b>{p:.2f}</b> {t}{warn}</div>'
    head = (f'<div style="{CARD}font-weight:600;">precision {m["precision"]:.2f} · recall {m["recall"]:.2f} · '
            f'missed {m["missed"]} · junk kept {m["junk"]}</div>')
    display(HTML(box(f'<div style="color:{SLATE};font-size:13px;">query: <b>{QUERY}</b></div>{head}'
                     f'<div style="{CARD}">{rows}</div>')))


widgets.interact(filter_board, threshold=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.05));

interactive(children=(FloatSlider(value=0.5, description='threshold', max=1.0, step=0.05), Output()), _dom_cla…

## Step 6 · Try it yourself

Edit the query and the passages (one per line) and press **Filter** (needs a live kernel). It uses `KEEP_AT`. Ideas:
- The default dog-on-a-train example → the two ticket rules are **KEEP**, the departure time and the dog-walking app are
  **drop**.
- Add `Bicycles may be taken on trains outside rush hour.` → same topic (taking things on trains), wrong subject:
  expect **drop**.
- Add `Assistance dogs always travel free.` → directly answers the question for some readers: expect **KEEP**.
- Then change the query to `How much does a dog ticket cost?` → the two ticket rules stay high, while the
  assistance-dog line usually drops towards the middle (it's about *who* pays, less about *how much*). Still above
  `KEEP_AT`, but it shows how the same passage scores differently for a different question.

In [9]:
def custom_html(query, passages):
    items = [ln.strip("- ").strip() for ln in passages.splitlines() if ln.strip()]
    kept_scores = usefulness(query, items, jev)
    rows = "".join(f'<div style="padding:3px 0;opacity:{1 if p >= KEEP_AT else .45};">'
                   f'{badge("KEEP", SAGE) if p >= KEEP_AT else badge("drop", "#B9B4A8")} <b>{p:.2f}</b> {t}</div>'
                   for t, p in zip(items, kept_scores))
    return box(f'<div style="{CARD}"><div style="color:{SLATE};font-size:13px;">{query}</div>{rows}</div>')


form_app([("query", "Query", "Can I bring my dog on the train?", 1),
          ("passages", "Passages (one per line)",
           "Small dogs in a carrier travel free on all regional trains.\n"
           "The train departs from platform 4 at 9:15.\n"
           "Large dogs need a half-price ticket and must wear a muzzle.\n"
           "Our dog-walking app connects you with trusted local walkers.", 5)],
         custom_html, "Filter")

Output()

## Recap and exercises

**Key ideas**
1. Retrieval matches words; a **usefulness gate** checks meaning. One `Noul` per passage, all in **one** Jev call.
2. A filter has two failure modes: **missed** useful passages (recall) and **junk kept** (precision). Measure both.
3. **Measure, then pick the threshold.** Here 0.5 looked sensible and dropped half the useful passages; the measurement
   showed a gap, and 0.2 sits inside it.
4. Put the threshold in the **gap**, leaning towards the cheaper mistake (in RAG, usually towards keeping more).

**Exercises**
- Set `KEEP_AT = 0.5` and re-run Step 4. Which facts disappear from the context?
- Write a second labelled query (for example *"How do I update my router's firmware?"* with 6 passages) and run
  `measure()` on both. Does 0.2 still sit in the gap?
- Reword the question to *"Is `passages[i]` relevant to someone trying to do what `query` asks?"* and re-measure. Do the
  two warning passages move up? (Wording is the model's whole world.)
- Combine with notebook 08: rerank first, then filter, and pass only the top 3 kept passages.